| Roll No.| Subject Code | Assignment No. |
|:-------:|:------------:|:--------------:|
| 2650026 | CS69101      | Lab 6          |

Given a set of integer elements stored in an input file, `input.txt`, construct a **Segment Tree** to efficiently perform **Range Minimum Query (RMQ)** and **Range Maximum Query** operations.

The Segment Tree should store both the minimum and maximum values corresponding to each segment of the input array. Implement range query and point update operations while maintaining the correctness of the Segment Tree.

In [1]:
import sys
from io import TextIOWrapper
from typing import Self, Any, Callable, TypeVar, Generic, Optional
import random
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pylab import figure

sys.setrecursionlimit(100000) # Fix to avoid RecursionError

prep_arr = [random.randint(1, (i+1)*10) for i in range(5000)] # Random Array of 5000 Elements
# prep_arr.sort() # Sorted Array for Worst Case Scenario
print(prep_arr[:25])
print(len(prep_arr))

# Write the array to a file
f = open("./2650026_lab6_data/input0.txt", "w")
for i in prep_arr:
    f.write(f"{i}, ")
f.write("\n")
f.close()

del prep_arr
del f

iterations = 0
PRINT_DEBUG = False

[9, 8, 7, 31, 30, 27, 5, 76, 12, 98, 17, 38, 59, 19, 62, 76, 87, 30, 13, 15, 86, 119, 81, 130, 20]
5000


# Q1.

Read integer elements from `input.txt` and construct a Segment Tree using the given array elements.

## File Read

In [2]:
# Read a set of integer elements from an input file (input.txt)
arr0 = []
data0 = open("./2650026_lab6_data/input.txt", 'r')

try:
    for line in data0.readlines():
        line_data = line.rstrip().rstrip(',').replace(' ','').split(',') #using rstrip to remove the \n
        for data in line_data:
            arr0.append(int(data))
    data0.close()
except Exception as e:
    sys.exit("[ERROR] Invalid data" + str(e))

In [3]:
print(arr0)

[70, 50, 90, 77, 55, 348, 76, 67, 69, 420, 25, 911, 210, 108, 768, 1080, 720]


## Segment Tree

In [4]:
# LEGACY CODE LOL

# class Segment_Node:
#     def __init__(self, value: int | None = None):
#         self.__lchild = None
#         self.__rchild = None
#         self.__value = None

#     def find_leaf(self) -> Self:
#         if self.__lchild is None and self.__rchild is None:
#             return self
#         elif self.__lchild is not None:
#             return self.__lchild.find_leaf()
#         else:
#             assert self.__rchild is not None # This is to shut the LSP up.
#             return self.__rchild.find_leaf()
    
#     def insert(self, value: int) -> Self: # Guess why this is wrong.
#         if self.__value is None:
#             self.__value = value
#         if self.__value == value:
#             return self
#         elif value < self.__value: #
#             if self.__lchild is None:
#                 new_node = type(self)(value) # type(self) instead of Segment_Node to shut the linter up.
#                 self.__lchild = (new_node)
#             else:
#                 self.__lchild.insert(value)
#         else:
#             if self.__rchild is None:
#                 new_node = type(self)(value)
#                 self.__rchild = new_node
#             else:
#                 self.__rchild.insert(value)
#         return self

In [9]:
MAX_DATSIZE = 10**7

class Segment_Tree:
    def __init__(self):
        self.__data = [0] * 4 * MAX_DATSIZE
        self.__size = 0
        self.__rangefn_name = "SUM"
        self.__rangefn = lambda x, y: x + y
        self.__active = [False] * MAX_DATSIZE


    def set_rangefn(self, rangefn_name: str, rangefn: Callable[[int, int], int] | None = None):
        self.__rangefn_name = rangefn_name
        if rangefn is not None:
            self.__rangefn = rangefn
        else:
            if rangefn_name == "SUM":
                self.__rangefn = lambda x, y: x + y
            elif rangefn_name == "MIN":
                self.__rangefn = lambda x, y: min(x, y)
            elif rangefn_name == "MAX":
                self.__rangefn = lambda x, y: max(x, y)
            else:
                raise ValueError(f"Invalid range function name: {rangefn_name}")
    
    def __has_active(self, index: int, left: int, right: int):
        if left == right:
            return self.__active[left]
        mid = (left + right) // 2
        return self.__has_active(index * 2 + 1, left, mid) or self.__has_active(index * 2 + 2, mid + 1, right)

    def insert(self, target_index: int, value: int, left_bound: int = 0, right_bound: int = MAX_DATSIZE - 1, index: int = 0):
        if left_bound == right_bound:
            if not self.__active[target_index]:
                self.__active[target_index] = True
                self.__size += 1
            self.__data[index] = value
            return
        mid = (left_bound + right_bound) // 2
        left_child_index = index * 2 + 1
        right_child_index = index * 2 + 2

        if target_index <= mid:
            self.insert(target_index, value, left_bound, mid, left_child_index)
        else:
            self.insert(target_index, value, mid + 1, right_bound, right_child_index)

        left_has = self.__has_active(left_child_index, left_bound, mid)
        right_has = self.__has_active(right_child_index, mid + 1, right_bound)

        if left_has and right_has:
            self.__data[index] = self.__rangefn(self.__data[left_child_index], self.__data[right_child_index])
        elif left_has:
            self.__data[index] = self.__data[left_child_index]
        elif right_has:
            self.__data[index] = self.__data[right_child_index]

    def delete(self, target_index: int, left_bound: int = 0, right_bound: int = MAX_DATSIZE - 1, index: int = 0):
        if not self.__has_active(index, left_bound, right_bound):
            return
        
        if left_bound == right_bound:
            if self.__active[target_index]:
                self.__active[target_index] = False
                self.__size -= 1
                self.__data[index] = 0
            return

        mid = (left_bound + right_bound) // 2
        left_child_index = index * 2 + 1
        right_child_index = index * 2 + 2

        if target_index <= mid:
            self.delete(target_index, left_bound, mid, left_child_index)
        else:
            self.delete(target_index, mid + 1, right_bound, right_child_index)

        left_has = self.__has_active(left_child_index, left_bound, mid)
        right_has = self.__has_active(right_child_index, mid + 1, right_bound)

        if left_has and right_has:
            self.__data[index] = self.__rangefn(self.__data[left_child_index], self.__data[right_child_index])
        elif left_has:
            self.__data[index] = self.__data[left_child_index]
        elif right_has:
            self.__data[index] = self.__data[right_child_index]
        else:
            self.__data[index] = 0

    def range_query(self, query_start: int, query_end: int, left_bound: int = 0, right_bound: int = MAX_DATSIZE - 1, index: int = 0) -> Optional[int]:
        if right_bound < query_start or query_end < left_bound or not self.__has_active(index, left_bound, right_bound):
            return None

        if query_start <= left_bound and right_bound <= query_end:
            return self.__data[index]

        mid = (left_bound + right_bound) // 2
        left_child_index = index * 2 + 1
        right_child_index = index * 2 + 2

        left_result = self.range_query(query_start, query_end, left_bound, mid, left_child_index)
        right_result = self.range_query(query_start, query_end, mid + 1, right_bound, right_child_index)

        if left_result is not None and right_result is not None:
            return self.__rangefn(left_result, right_result)
        elif left_result is not None:
            return left_result
        else:
            return right_result

In [10]:
stree0 = Segment_Tree()
stree0.set_rangefn("SUM")